In [0]:
files = {
    "employee" : "/Volumes/workspace/default/assignment/employees_extended.json",
    "Transaction" : "/Volumes/workspace/default/assignment/transactions_1200 (1).json"
}
df = {}

for name, path in files.items():
    df[name]=spark.read.json(
        path
    )

In [0]:
df["employee"].display()

address,age,department,emp_id,is_active,join_date,metadata,name,projects,salary,skills
"List(Hyderabad, TS)",26,Operations,1,false,2024-02-12,"List(L3, null, manual)",Ravi K,"List(List(35, Alpha), List(131, Alpha), List(278, Beta))",59378.66,List(GCP)
"List(Bangalore, KA)",30,Sales,2,true,2021-10-25,"List(L2, null, ingest)",Anu S,List(),null,"List(Spark, Scala, Excel)"
"List(Hyderabad, TS)",55,IT,3,true,2025-01-18,"List(null, null, api)",Anu V,List(),146639.47,"List(GCP, Scala, SQL)"
"List(Delhi, DL)",42,Operations,4,true,2020-12-29,"List(L3, null, api)",Kiran N,"List(List(393, Epsilon))",null,List(GCP)
"List(Hyderabad, TS)",22,Support,5,true,2022-07-02,"List(null, IN, null)",Divya B,"List(List(355, Beta), List(222, Delta))",121999.99,"List(SQL, Scala, GCP)"
"List(Hyderabad, TS)",57,Sales,6,false,2020-06-17,"List(null, IN, null)",Vikram V,List(),118376.73,"List(Azure, AWS)"
"List(Kochi, KL)",36,Finance,7,false,2018-02-17,"List(L3, null, ingest)",Ravi J,"List(List(348, Omega), List(77, Gamma))",99486.05,List(Tableau)
"List(Kochi, KL)",39,Marketing,8,false,2023-09-11,"List(null, null, null)",Karthik M,"List(List(418, Epsilon))",139078.48,"List(Spark, Excel)"
"List(Chennai, TN)",39,Sales,9,false,2020-09-13,"List(null, EU, api)",Ravi C,"List(List(55, Omega), List(292, Omega), List(84, Omega))",134840.23,List(SQL)
"List(Bangalore, KA)",39,Support,10,true,2025-07-14,"List(L3, null, ingest)",Manoj V,List(),116241.38,"List(AWS, Azure, Airflow)"


### Q6: Salary Banding

In [0]:
from pyspark.sql.functions import col,when

df["employee"].withColumn(
    "salary_band",
    when(col("salary")>80000,"High").
    when(
        (col("salary")>40000) & (col("salary")<=80000),"Medium"
        ).
    otherwise("Low")
).display()

address,age,department,emp_id,is_active,join_date,metadata,name,projects,salary,skills,salary_band
"List(Hyderabad, TS)",26,Operations,1,false,2024-02-12,"List(L3, null, manual)",Ravi K,"List(List(35, Alpha), List(131, Alpha), List(278, Beta))",59378.66,List(GCP),Medium
"List(Bangalore, KA)",30,Sales,2,true,2021-10-25,"List(L2, null, ingest)",Anu S,List(),null,"List(Spark, Scala, Excel)",Low
"List(Hyderabad, TS)",55,IT,3,true,2025-01-18,"List(null, null, api)",Anu V,List(),146639.47,"List(GCP, Scala, SQL)",High
"List(Delhi, DL)",42,Operations,4,true,2020-12-29,"List(L3, null, api)",Kiran N,"List(List(393, Epsilon))",null,List(GCP),Low
"List(Hyderabad, TS)",22,Support,5,true,2022-07-02,"List(null, IN, null)",Divya B,"List(List(355, Beta), List(222, Delta))",121999.99,"List(SQL, Scala, GCP)",High
"List(Hyderabad, TS)",57,Sales,6,false,2020-06-17,"List(null, IN, null)",Vikram V,List(),118376.73,"List(Azure, AWS)",High
"List(Kochi, KL)",36,Finance,7,false,2018-02-17,"List(L3, null, ingest)",Ravi J,"List(List(348, Omega), List(77, Gamma))",99486.05,List(Tableau),High
"List(Kochi, KL)",39,Marketing,8,false,2023-09-11,"List(null, null, null)",Karthik M,"List(List(418, Epsilon))",139078.48,"List(Spark, Excel)",High
"List(Chennai, TN)",39,Sales,9,false,2020-09-13,"List(null, EU, api)",Ravi C,"List(List(55, Omega), List(292, Omega), List(84, Omega))",134840.23,List(SQL),High
"List(Bangalore, KA)",39,Support,10,true,2025-07-14,"List(L3, null, ingest)",Manoj V,List(),116241.38,"List(AWS, Azure, Airflow)",High


### Q7: Active Employee Filter

In [0]:
from pyspark.sql.functions import expr

df["employee"] = df["employee"].withColumn(
    "join_date",
    expr("try_cast(join_date as date)")
    )

In [0]:
from pyspark.sql.functions import year,col
df["employee"].filter(
    (col("is_active") == "true") & (year(col("join_date"))==2022)
    ).select("emp_id","name","is_active","join_date").display()

emp_id,name,is_active,join_date
5,Divya B,true,2022-07-02
101,Priya T,true,2022-12-12
113,Sana P,true,2022-04-16
120,Manoj A,true,2022-01-11
125,Rahul B,true,2022-04-03
150,Meena N,true,2022-02-14
155,Aisha J,true,2022-11-15
172,Vikram H,true,2022-03-30
178,John N,true,2022-02-28
193,Karthik L,true,2022-10-24


### Q8: Null Handling 

In [0]:
df["employee"].filter(
    col("salary").isNull()
).select("name").display()

name


In [0]:
from pyspark.sql.functions import *

avg_Salary_dept = df["employee"].groupBy("department").agg(avg("salary").alias("avg_salary"))
display(avg_Salary_dept)

department,avg_salary
Support,92315.66154761902
Sales,95739.51616279075
IT,78836.20866666664
Finance,87073.30875
Marketing,88937.39559523811
HR,86691.9509090909
Operations,90613.16612499999


In [0]:
from pyspark.sql.functions import expr

df["employee"] = df["employee"].withColumn(
    "salary",
    expr("try_cast(salary as float)")
    )

In [0]:
from pyspark.sql.functions import col,when
from pyspark.sql.window import Window

window_spec = Window.partitionBy("department")

df["employee"] = df["employee"].withColumn(
    "avg_Salary_dept",
    avg("salary").over(window_spec)
)

df["employee"] = df["employee"].withColumn(
    "salary",
    when(
        col("salary").isNull(),
        col("avg_Salary_dept")
        ).otherwise(col("salary")
    )
).drop("avg_Salary_dept")


df["employee"].display()

address,age,department,emp_id,is_active,join_date,metadata,name,projects,salary,skills
"List(Kochi, KL)",36,Finance,7,false,2018-02-17,"List(L3, null, ingest)",Ravi J,"List(List(348, Omega), List(77, Gamma))",99486.046875,List(Tableau)
"List(Mumbai, MH)",null,Finance,12,false,2021-10-21,"List(L2, IN, api)",John J,"List(List(294, Beta))",87073.30864257812,List(Scala)
"List(Kochi, KL)",24,Finance,33,false,2025-01-13,"List(null, EU, null)",Rahul C,List(),74581.8125,"List(SQL, Java)"
"List(Bangalore, KA)",53,Finance,38,true,2019-02-16,"List(L1, null, api)",Arun H,"List(List(174, Beta), List(382, Alpha))",59562.44921875,"List(Java, Scala)"
"List(Hyderabad, TS)",38,Finance,39,false,2022-12-10,"List(L1, null, manual)",Priya L,"List(List(440, Alpha), List(225, Alpha), List(57, Delta))",87073.30864257812,"List(Spark, Python, Java)"
"List(Delhi, DL)",null,Finance,47,false,2024-09-21,"List(L3, null, manual)",Priya N,"List(List(498, Gamma), List(279, Omega), List(440, Gamma))",55466.359375,"List(SQL, Java)"
"List(Kochi, KL)",49,Finance,52,false,2023-02-26,"List(null, US, api)",Rahul A,"List(List(299, Zeta), List(57, Beta))",87073.30864257812,"List(Tableau, Python, AWS)"
"List(Chennai, TN)",55,Finance,57,true,2024-11-03,"List(L2, IN, ingest)",Rahul A,"List(List(174, Zeta), List(229, Zeta), List(91, Alpha))",28128.080078125,"List(Tableau, SQL)"
"List(Chennai, TN)",null,Finance,62,false,2018-11-27,"List(null, EU, manual)",Anu B,"List(List(339, Gamma))",87073.30864257812,"List(GCP, Spark, SQL)"
"List(Delhi, DL)",55,Finance,63,false,2020-05-30,"List(L1, null, api)",Vikram S,"List(List(232, Delta), List(68, Zeta), List(238, Gamma))",37397.30859375,"List(AWS, Scala, PowerBI, Excel)"
